In [1]:
import os
import numpy as np
import cv2
from PIL import Image, ImageStat
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
def estimate_jpeg_quality(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, laplacian = cv2.threshold(cv2.convertScaleAbs(cv2.Laplacian(gray, 3)), 0, 255, cv2.THRESH_BINARY)
    return np.mean(laplacian)


In [3]:
def extract_features(image_path):
    try:
        img = Image.open(image_path)
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            print(f"Warning: Unable to read {image_path} with OpenCV. Skipping...")
            return None
    except:
        print(f"Error: Unable to open {image_path}. Skipping...")
        return None
    
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    features = []
    
    # 1. Estimated JPEG Quality
    jpeg_quality = estimate_jpeg_quality(img_cv)
    features.append(jpeg_quality)
    
    # 2. Image sharpness (using variance of Laplacian)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    features.append(sharpness)
    
    # 3. RGB channel statistics
    stat = ImageStat.Stat(img)
    for channel in range(3):  # R, G, B
        features.extend([stat.mean[channel], stat.rms[channel], stat.var[channel]])
    
    # 4. Image entropy (measure of image complexity)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-10))
    features.append(entropy)
    
    # 5. Edge density (using Sobel operator)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_density = np.mean(np.sqrt(sobel_x**2 + sobel_y**2))
    features.append(edge_density)
    
    # 6. Color Range (max - min) for each channel
    for i in range(3):
        channel = img_cv[:,:,i]
        features.append(np.max(channel) - np.min(channel))
    
    return features

In [4]:
# Paths to your dataset
real_path = "C://Users//Sinchan A//Desktop//Internship//vid//real"
fake_path = "C://Users//Sinchan A//Desktop//Internship//vid//fake"


In [5]:
# Lists to store features and labels
X = []
y = []

# Function to safely add features
def add_features(img_path, label):
    features = extract_features(img_path)
    if features is not None:
        X.append(features)
        y.append(label)

In [6]:
# Extract features
for img_name in os.listdir(real_path):
    img_path = os.path.join(real_path, img_name)
    add_features(img_path, 0)  # 0 for real

for img_name in os.listdir(fake_path):
    img_path = os.path.join(fake_path, img_name)
    add_features(img_path, 1)  # 1 for fake

In [7]:
# Convert to numpy arrays (only if we have data)
if X and y:
    X = np.array(X)
    y = np.array(y)

    # Scale the features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    from imblearn.over_sampling import SMOTE
    from collections import Counter

    counter = Counter(y_train)
    print('Before', counter)

 # oversampling the train dataset using SMOTE
    smt = SMOTE()
    X_train, y_train = smt.fit_resample(X_train, y_train)

    counter = Counter(y_train)
    print('After', counter)

    # Initialize base models
    svm = SVC(probability=True)  # Set probability=True for predict_proba()
    rf = RandomForestClassifier(n_estimators=100, random_state=42)

    # Train base models
    svm.fit(X_train, y_train)
    rf.fit(X_train, y_train)

    # Get predictions from base models
    svm_preds = svm.predict_proba(X_train)
    rf_preds = rf.predict_proba(X_train)

    # Stack predictions into a new feature set
    stacked_X = np.column_stack((svm_preds, rf_preds))

    # Initialize meta-classifier
    meta_clf = LogisticRegression()

    # Train meta-classifier on base model predictions
    meta_clf.fit(stacked_X, y_train)

    # Make predictions on test set using the stacked model
    base_svm_preds = svm.predict_proba(X_test)
    base_rf_preds = rf.predict_proba(X_test)
    stacked_X_test = np.column_stack((base_svm_preds, base_rf_preds))
    y_pred = meta_clf.predict(stacked_X_test)
      # Evaluate the stacked model
    print("\nTest Set Performance (Stacked Model):")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(f"  Precision (Macro): {precision_score(y_test, y_pred, average='macro'):.3f}")
    print(f"  Recall (Macro): {recall_score(y_test, y_pred, average='macro'):.3f}")
    print(f"  F1-Score (Macro): {f1_score(y_test, y_pred, average='macro'):.3f}")
    print(f"  Precision (Weighted): {precision_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"  Recall (Weighted): {recall_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"  F1-Score (Weighted): {f1_score(y_test, y_pred, average='weighted'):.3f}")



Before Counter({0: 3229, 1: 3036})
After Counter({1: 3229, 0: 3229})

Test Set Performance (Stacked Model):
  Accuracy: 1.000
  Precision (Macro): 1.000
  Recall (Macro): 1.000
  F1-Score (Macro): 1.000
  Precision (Weighted): 1.000
  Recall (Weighted): 1.000
  F1-Score (Weighted): 1.000
